In [25]:
from IPython import display

In [26]:
try:
    import nbformat
except:
    !pip install nbformat
display.clear_output()

In [27]:
%run ../model/train.ipynb

+ action: train_feat01
+ feat_path: ../../exps/featbase_29112025/data.npz
+ seed: 42
+ exp_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\process\exps
+ exp_name: train_29112025
+ data_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\data
+ verbose: True
+ save_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\process\exps\train_29112025
---------- information ----------
train-col: {'valence', 'acousticness', 'Id', 'time_signature', 'key', 'danceability', 'Track Name', 'speechiness', 'loudness', 'Popularity', 'instrumentalness', 'mode', 'energy', 'liveness', 'tempo', 'duration_in min/ms', 'Class', 'Artist Name'}
test-col: {'valence', 'acousticness', 'Id', 'time_signature', 'key', 'danceability', 'Track Name', 'speechiness', 'loudness', 'Popularity', 'instrumentalness', 'mode', 'energy', 'liveness', 'tempo', 'duration_in min/ms', 'Artist Name'}
Union: {'valence', 'time_signature', 'acousticness', 'Id', 'key', 'danceability', 'Track Name', 'speechi

In [28]:
!jupyter nbconvert  ../model/train.ipynb --to html 

[NbConvertApp] Converting notebook ../model/train.ipynb to html
[NbConvertApp] Writing 312156 bytes to ..\model\train.html


# Import lib


In [29]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
import joblib

date = "29112025"

data = np.load(f"../../exps/train_{date}/data.npz", allow_pickle=True)

In [30]:
with np.load(f'../../exps/train_{date}/data.npz') as z:
    print(z.files)          # shows the list of keys present

['train_data', 'test_data', 'train_columns', 'test_columns']


In [31]:
data['train_columns']

array(['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'duration_in min/ms', 'time_signature',
       'acoustic_energy', 'loudness_energy', 'Output'], dtype=object)

In [32]:
# Lấy dữ liệu train và test
df_train = pd.DataFrame(data = data['train_data'], columns= data['train_columns'])
df_test = pd.DataFrame(data['test_data'], columns=df_train.drop(columns=['Output']).columns)


In [33]:
df_train

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,acoustic_energy,loudness_energy,Output
0,37.0,0.334,0.536,9.0,-6.649,0.0,0.0381,0.378000,0.035449,0.1060,0.235,152.429,204947.000000,4.0,0.202608,1.090532,9.0
1,67.0,0.725,0.747,11.0,-5.545,1.0,0.0876,0.027200,0.046800,0.1040,0.380,132.921,191956.000000,4.0,0.020318,1.403390,6.0
2,44.0,0.584,0.804,7.0,-6.094,1.0,0.0619,0.000968,0.635000,0.2840,0.635,159.953,161037.000000,4.0,0.000778,1.575236,10.0
3,12.0,0.515,0.308,6.0,-14.711,1.0,0.0312,0.907000,0.021300,0.3000,0.501,172.472,298093.000000,3.0,0.279356,0.848343,2.0
4,48.0,0.565,0.777,6.0,-5.096,0.0,0.2490,0.183000,0.000482,0.2110,0.619,88.311,254145.000000,4.0,0.142191,1.404531,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14391,47.0,0.607,0.946,1.0,-2.965,1.0,0.1500,0.005480,0.000390,0.2780,0.653,120.011,195181.000000,4.0,0.005184,1.303121,10.0
14392,27.0,0.435,0.951,8.0,-7.475,1.0,0.0576,0.000005,0.550000,0.0952,0.203,135.034,282043.000000,4.0,0.000005,2.032402,8.0
14393,22.0,0.415,0.941,11.0,-4.300,1.0,0.0524,0.001810,0.000004,0.3370,0.572,167.978,176529.000000,4.0,0.001703,1.569312,10.0
14394,37.0,0.493,0.986,1.0,-2.279,1.0,0.0917,0.000967,0.006620,0.1230,0.567,122.036,186307.000000,4.0,0.000953,1.170913,10.0


In [34]:
df_train.isna().sum().sum()

np.int64(0)

In [35]:
df_test

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,acoustic_energy,loudness_energy
0,44.0,0.691,0.670,2.0,-7.093,0.0,0.0941,0.075700,0.035200,0.1970,0.635,89.965,200000.0,4.0,0.050719,1.400970
1,14.0,0.461,0.777,2.0,-7.469,1.0,0.0306,0.388000,0.923000,0.2910,0.525,163.043,283909.0,4.0,0.301476,1.659992
2,80.0,0.656,0.291,2.0,-10.572,1.0,0.0293,0.872000,0.324778,0.1140,0.298,103.971,232533.0,4.0,0.253752,0.712539
3,52.0,0.480,0.826,6.0,-4.602,1.0,0.0397,0.000797,0.000001,0.1250,0.687,96.000,222053.0,4.0,0.000658,1.423300
4,23.0,0.734,0.729,1.0,-6.381,0.0,0.2830,0.147000,0.353426,0.0672,0.805,76.030,118439.0,4.0,0.107163,1.457205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3595,55.0,0.271,0.866,5.0,-4.072,0.0,0.0578,0.048900,0.000762,0.1160,0.127,175.665,267987.0,4.0,0.042347,1.406155
3596,38.0,0.598,0.690,5.0,-4.758,0.0,0.3030,0.363000,0.057252,0.0519,0.653,187.934,198300.0,4.0,0.250470,1.207907
3597,18.0,0.783,0.820,1.0,-6.102,1.0,0.0489,0.000540,0.499000,0.0628,0.235,129.015,339213.0,4.0,0.000443,1.607509
3598,38.0,0.443,0.401,5.0,-13.997,1.0,0.0426,0.263000,0.161750,0.1680,0.751,168.209,182587.0,3.0,0.105463,1.085848


In [36]:
df_test.isna().sum().sum()

np.int64(0)

In [37]:
X_train = df_train.drop(columns=['Output'], axis=1)
y_train = df_train['Output']

In [38]:
X_test = df_test

# Logistic Regression


In [39]:
# Tạo pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

# Thiết lập hyperparameters
param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2'],
    'logreg__solver': ['liblinear']
}

# Sử dụng Stratified K-Fold
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# Grid Search với multiple scoring metrics
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Chọn metric để chọn best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nBắt đầu training với GridSearch...")
grid_search.fit(X_train, y_train)

# Kết quả
print("\nBest parameters:", grid_search.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search.best_score_))

# Lấy tất cả kết quả cross-validation|
cv_results = grid_search.cv_results_
print("\nCross-validation results:")
for i in range(len(cv_results['params'])):
    print(f"Params: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    # print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    # print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")


Bắt đầu training với GridSearch...
Fitting 2 folds for each of 10 candidates, totalling 20 fits



Best parameters: {'logreg__C': 100, 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
Best cross-validation accuracy: 0.4887

Cross-validation results:
Params: {'logreg__C': 0.01, 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4420 (+/- 0.0047)
Params: {'logreg__C': 0.01, 'logreg__penalty': 'l2', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4656 (+/- 0.0070)
Params: {'logreg__C': 0.1, 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4842 (+/- 0.0042)
Params: {'logreg__C': 0.1, 'logreg__penalty': 'l2', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4815 (+/- 0.0038)
Params: {'logreg__C': 1, 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4876 (+/- 0.0056)
Params: {'logreg__C': 1, 'logreg__penalty': 'l2', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4868 (+/- 0.0051)
Params: {'logreg__C': 10, 'logreg__penalty': 'l1', 'logreg__solver': 'liblinear'}
  Accuracy: 0.4885 (+/- 0.0044)
Params: {'logreg__C': 10, 'logreg__pena

In [40]:
# Model tốt nhất
best_model_lr = grid_search.best_estimator_

# Đánh giá trên training set
y_pred_train = best_model_lr.predict(X_train)
y_pred_proba_train = best_model_lr.predict_proba(X_train)[:, 1]  # Xác suất lớp 1

# Tính các metrics
train_accuracy = best_model_lr.score(X_train, y_train)
# train_f1 = f1_score(y_train, y_pred_train)
# train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)

print(f"\n=== KẾT QUẢ TRÊN TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
# print(f"F1 Score: {train_f1:.4f}")
# print(f"ROC AUC: {train_roc_auc:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_train, y_pred_train))


=== KẾT QUẢ TRÊN TRAINING SET ===
Accuracy: 0.4903


## Dump mô hình logistic regression


In [41]:
import joblib

# Lưu model
# os.makedirs(f'../../exps/trainbase_{date}', exist_ok=True)
joblib.dump(best_model_lr, f'../../exps/train_{date}/logistic_regression_model.pkl')
print("Model đã được lưu bằng joblib")

# Tham số
feature_names = X_train.columns.tolist()
scaler = best_model_lr.named_steps['scaler']

# Lưu nhiều objects cùng lúc
joblib.dump({
    'model': best_model_lr,
    'feature_names': feature_names,  # Nếu có
    'scaler': scaler,  # Nếu có
    'training_date': f'{date}'
}, f'../../exps/train_{date}/model_package.pkl')

# Load lại model
loaded_model = joblib.load(f'../../exps/train_{date}/logistic_regression_model.pkl')

Model đã được lưu bằng joblib


# Random forest


In [42]:
# Tạo pipeline cho Random Forest
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),  # Vẫn có thể chuẩn hóa cho RF (không bắt buộc)
    ('rf', RandomForestClassifier(random_state=42))
])

# Thiết lập hyperparameters cho Random Forest
param_grid_rf = {
    'rf__n_estimators': [50, 100, 200],           # Số cây trong rừng
    'rf__max_depth': [3, 5, 7],             # Độ sâu tối đa của cây
    'rf__min_samples_split': [2, 5, 10],          # Số mẫu tối thiểu để split
    'rf__min_samples_leaf': [1, 2, 4],            # Số mẫu tối thiểu ở lá
    'rf__max_features': ['sqrt', 'log2']    # Số features xét tại mỗi split
}

# Sử dụng Stratified K-Fold
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# Grid Search với multiple scoring metrics
grid_search_rf = GridSearchCV(
    pipeline_rf, 
    param_grid_rf, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Chọn metric để chọn best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nBắt đầu training Random Forest với GridSearch...")
grid_search_rf.fit(X_train, y_train)

# Kết quả
print("\nBest parameters:", grid_search_rf.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search_rf.best_score_))

# Lấy tất cả kết quả cross-validation
cv_results = grid_search_rf.cv_results_
print("\nCross-validation results (top 5 combinations):")
# Chỉ hiển thị 5 combination tốt nhất
best_indices = np.argsort(cv_results['mean_test_accuracy'])[-5:][::-1]

for i in best_indices:
    print(f"\nParams: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    # print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    # print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")

# Model tốt nhất
best_model_rf = grid_search_rf.best_estimator_

# Đánh giá trên training set
y_pred_train = best_model_rf.predict(X_train)
y_pred_proba_train = best_model_rf.predict_proba(X_train)[:, 1]  # Xác suất lớp 1

# Tính các metrics
train_accuracy = best_model_rf.score(X_train, y_train)
# train_f1 = f1_score(y_train, y_pred_train)
# train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)


Bắt đầu training Random Forest với GridSearch...
Fitting 2 folds for each of 162 candidates, totalling 324 fits

Best parameters: {'rf__max_depth': 7, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 10, 'rf__n_estimators': 50}
Best cross-validation accuracy: 0.4971

Cross-validation results (top 5 combinations):

Params: {'rf__max_depth': 7, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 10, 'rf__n_estimators': 50}
  Accuracy: 0.4971 (+/- 0.0029)

Params: {'rf__max_depth': 7, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 10, 'rf__n_estimators': 50}
  Accuracy: 0.4971 (+/- 0.0029)

Params: {'rf__max_depth': 7, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 4, 'rf__min_samples_split': 2, 'rf__n_estimators': 50}
  Accuracy: 0.4959 (+/- 0.0017)

Params: {'rf__max_depth': 7, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 4, 'rf__min_samples_split': 5, 'rf__n_estimators': 50}
  Accur

In [43]:
print(f"\n=== KẾT QUẢ TRÊN TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
# print(f"F1 Score: {train_f1:.4f}")
# print(f"ROC AUC: {train_roc_auc:.4f}")



=== KẾT QUẢ TRÊN TRAINING SET ===
Accuracy: 0.5287


# Dump mô hình Random Forest


In [44]:
import joblib

# Lưu model
joblib.dump(best_model_rf, f'../../exps/train_{date}/random_forest_model.pkl')
print("Model đã được lưu bằng joblib")

# Tham số
feature_names = X_train.columns.tolist()
scaler = best_model_rf.named_steps['scaler']

# Lưu nhiều objects cùng lúc
joblib.dump({
    'model': best_model_rf,
    'feature_names': feature_names,  # Nếu có
    'scaler': scaler,  # Nếu có
    'training_date': f'{date}'
}, f'../../exps/train_{date}/model_package.pkl')

# Load lại model
loaded_model = joblib.load(f'../../exps/train_{date}/random_forest_model.pkl')

Model đã được lưu bằng joblib


# XGBoost


In [45]:
# Tạo pipeline cho XGBoost
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),  # XGBoost thường không cần scaling nhưng vẫn có thể dùng
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# Thiết lập hyperparameters cho XGBoost
param_grid_xgb = {
    'xgb__n_estimators': [50, 100],           # Số cây (boosting rounds)
    'xgb__max_depth': [3, 5],                   # Độ sâu tối đa của cây
    'xgb__learning_rate': [0.01, 0.1],       # Tốc độ học
    'xgb__subsample': [0.8, 0.9],            # Tỷ lệ mẫu cho mỗi cây
    'xgb__colsample_bytree': [0.8, 0.9],     # Tỷ lệ features cho mỗi cây
    'xgb__reg_alpha': [0, 0.1],                # L1 regularization
    'xgb__reg_lambda': [1, 10]               # L2 regularization
}

# Sử dụng Stratified K-Fold
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# Grid Search với multiple scoring metrics
grid_search_xgb = GridSearchCV(
    pipeline_xgb, 
    param_grid_xgb, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Chọn metric để chọn best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nBắt đầu training XGBoost với GridSearch...")
grid_search_xgb.fit(X_train, y_train)

# Kết quả
print("\nBest parameters:", grid_search_xgb.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search_xgb.best_score_))

# Lấy tất cả kết quả cross-validation
cv_results = grid_search_xgb.cv_results_
print("\nCross-validation results (top 5 combinations):")
# Chỉ hiển thị 5 combination tốt nhất
best_indices = np.argsort(cv_results['mean_test_accuracy'])[-5:][::-1]

for i in best_indices:
    print(f"\nParams: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    # print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    # print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")

# Model tốt nhất
best_model_xgb = grid_search_xgb.best_estimator_

# Đánh giá trên training set
y_pred_train = best_model_xgb.predict(X_train)
y_pred_proba_train = best_model_xgb.predict_proba(X_train)[:, 1]  # Xác suất lớp 1

# Tính các metrics
train_accuracy = best_model_xgb.score(X_train, y_train)
# train_f1 = f1_score(y_train, y_pred_train)
# train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)


Bắt đầu training XGBoost với GridSearch...
Fitting 2 folds for each of 128 candidates, totalling 256 fits

Best parameters: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 5, 'xgb__n_estimators': 100, 'xgb__reg_alpha': 0.1, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.9}
Best cross-validation accuracy: 0.5323

Cross-validation results (top 5 combinations):

Params: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 5, 'xgb__n_estimators': 100, 'xgb__reg_alpha': 0.1, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.9}
  Accuracy: 0.5323 (+/- 0.0002)

Params: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 5, 'xgb__n_estimators': 100, 'xgb__reg_alpha': 0, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.9}
  Accuracy: 0.5313 (+/- 0.0000)

Params: {'xgb__colsample_bytree': 0.9, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 5, 'xgb__n_estimators': 100, 'xgb__reg_alpha': 0, 'xgb__reg_lambda': 10, 'xgb__subsample': 0.9}
  Acc

In [46]:

print(f"\n=== KẾT QUẢ TRÊN TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
# print(f"F1 Score: {train_f1:.4f}")
# print(f"ROC AUC: {train_roc_auc:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_train, y_pred_train))


=== KẾT QUẢ TRÊN TRAINING SET ===
Accuracy: 0.6321


# Dump mô hình XGBoost


In [47]:
import joblib

# Lưu model
joblib.dump(best_model_xgb, f'../../exps/train_{date}/xgboost_model.pkl')
print("Model đã được lưu bằng joblib")

# Tham số
feature_names = X_train.columns.tolist()
scaler = best_model_xgb.named_steps['scaler']

# Lưu nhiều objects cùng lúc
joblib.dump({
    'model': best_model_xgb,
    'feature_names': feature_names,  # Nếu có
    'scaler': scaler,  # Nếu có
    'training_date': f'{date}'
}, f'../../exps/train_{date}/model_package.pkl')

# Load lại model
loaded_model = joblib.load(f'../../exps/train_{date}/xgboost_model.pkl')

Model đã được lưu bằng joblib


# Chọn ra mô hình tối ưu


In [49]:
# Sử dụng best model từ GridSearch
best_model = best_model_rf

# Kiểm tra model đã được train chưa
print("Model đã được train:", hasattr(best_model.named_steps['rf'], 'feature_importances_'))

# Dự đoán trên test set
test_preds = best_model.predict(X_test)
test_probas = best_model.predict_proba(X_test)[:, 1]  # Nếu cần xác suất

print(f"Số lượng predictions: {len(test_preds)}")
print(f"Phân bố predictions: {pd.Series(test_preds).value_counts().sort_index()}")

test_data_orig = pd.read_csv("../../../data/test.csv")

# Tạo submission
submission = pd.DataFrame({
    "Id": test_data_orig["Id"],
    "Class": test_preds.astype(int)
})

# Lưu submission
submission.to_csv(f"../../exps/train_{date}/random_forest.csv", index=False)
print("Submission file đã được lưu!")

# Kiểm tra submission
print("\n=== SUBMISSION INFO ===")
print(f"Shape: {submission.shape}")
print(f"Id range: {submission['Id'].min()} - {submission['Id'].max()}")
print(f"Class distribution:\n{submission['Class'].value_counts().sort_index()}")

Model đã được train: True
Số lượng predictions: 3600
Phân bố predictions: 0.0      187
2.0       29
3.0       65
4.0       40
5.0      286
6.0      186
7.0      108
8.0      265
9.0      537
10.0    1897
Name: count, dtype: int64
Submission file đã được lưu!

=== SUBMISSION INFO ===
Shape: (3600, 2)
Id range: 14397 - 17996
Class distribution:
Class
0      187
2       29
3       65
4       40
5      286
6      186
7      108
8      265
9      537
10    1897
Name: count, dtype: int64
